# Fine-tuning e comparação de transformers

Compara os **cinco modelos** de `compare_transformers.ipynb` antes e depois do fine-tuning, usando os mesmos emails de validação. Sem fine-tuning, os quatro encoders usam semelhança entre embeddings e descrições das classes; o XLM-RoBERTa-large-XNLI usa entailment, como no notebook original. Esses métodos diferem do classificador supervisionado após o treino. Com apenas 26 emails e um único **Pedido de Encomenda**, esse exemplo fica obrigatoriamente no treino; a validação não mede essa classe. Os resultados são exploratórios. A regra de produção `SPAM` no assunto **não** é aplicada aqui: o objetivo é comparar os transformers isoladamente e existem três emails marcados `SPAM` no assunto com label humana `Pedido de Informação`.

Executa a partir da raiz ou da pasta `research/`, com o ambiente `.GB` selecionado. Cada modelo pode demorar bastante e usar vários GB de memória; ajusta `MODELS` para uma execução parcial. Nenhum checkpoint de produção é alterado.

In [18]:
import gc
import json
import random
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import (AutoConfig, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/email_data.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Abre o notebook dentro do projeto GlobalBrico.')
sys.path.insert(0, str(ROOT))
from src.email_data import LABELS, load_labelled_emails

MODELS = [
    ('NorBERTo-large', 'Itau-Unibanco/NorBERTo-large'),
    ('XLM-RoBERTa-base', 'FacebookAI/xlm-roberta-base'),
    ('XLM-RoBERTa-large-XNLI', 'joeddav/xlm-roberta-large-xnli'),
    ('Albertina 900M PT-PT', 'PORTULAN/albertina-900m-portuguese-ptpt-encoder'),
    ('BERTimbau Base', 'neuralmind/bert-base-portuguese-cased'),
]

SEED = 42
EPOCHS = 3
MAX_LENGTH = 256
BATCH_SIZE = 1
LEARNING_RATE = 2e-5
RUN_DIR = ROOT / 'research/finetuning_comparison' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
set_seed(SEED)
print('Resultados:', RUN_DIR)

Resultados: /home/alex/Desktop/GlobalBrico/research/finetuning_comparison/20260914T150516Z


In [19]:
examples = load_labelled_emails(
    ROOT / 'research/ground_truth/ground_truth_emails.xlsx',
    ROOT / 'research/extracted_emails',
)
# A classe com um único exemplo permanece no treino. Para as restantes, a mesma
# amostra determinística de cerca de 25% serve de validação para todos os modelos.
rng = random.Random(SEED)
by_label = {label: [i for i, item in enumerate(examples) if item['label'] == label] for label in LABELS}
val_indices = []
for label, indices in by_label.items():
    indices = indices.copy()
    rng.shuffle(indices)
    if len(indices) > 1:
        val_indices.extend(indices[:max(1, round(len(indices) * 0.25))])
val_indices = sorted(val_indices)
train_indices = [i for i in range(len(examples)) if i not in set(val_indices)]
train_examples = [examples[i] for i in train_indices]
val_examples = [examples[i] for i in val_indices]
assert set(item['uid'] for item in train_examples).isdisjoint(item['uid'] for item in val_examples)
assert set(item['label'] for item in train_examples) == set(LABELS)
print('Treino:', Counter(item['label'] for item in train_examples))
print('Validação:', Counter(item['label'] for item in val_examples))
(RUN_DIR / 'split.json').write_text(json.dumps({
    'seed': SEED, 'train_uids': [x['uid'] for x in train_examples],
    'validation_uids': [x['uid'] for x in val_examples],
}, ensure_ascii=False, indent=2), encoding='utf-8')

Treino: Counter({'Pedido de Informação': 13, 'SPAM': 6, 'Pedido de Encomenda': 1})
Validação: Counter({'Pedido de Informação': 4, 'SPAM': 2})


403

In [20]:
from transformers import AutoModel

HYPOTHESES = [
    'Este email solicita informações, preços, um orçamento ou esclarecimentos sobre produtos ou serviços.',
    'Este email faz ou confirma uma encomenda de produtos ou serviços.',
    'Este email não solicita informações nem faz uma encomenda; é outra correspondência ou spam.',
]

# Usa a GPU NVIDIA quando disponível, para o XLM-RoBERTa e demais modelos.
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    DEVICE = torch.device('cuda')
    print(f'Usando GPU CUDA: {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu')
    print('CUDA indisponível; a execução vai usar CPU.')




Usando GPU CUDA: NVIDIA GB10


In [21]:
def baseline_predict(repo, examples):
    start = time.perf_counter()
    texts = [item['text'] for item in examples]
    config = AutoConfig.from_pretrained(repo, trust_remote_code=False)

    if repo == 'joeddav/xlm-roberta-large-xnli':
        tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=False)
        load_options = {}
        model = AutoModelForSequenceClassification.from_pretrained(
            repo, config=config, trust_remote_code=False, **load_options
        ).to(DEVICE).eval()
        entailment = [i for i, label in config.id2label.items() if str(label).lower().startswith('entail')]
        if len(entailment) != 1:
            raise ValueError(f'Configuração NLI sem label entailment inequívoca: {config.id2label}')

        all_scores = []
        with torch.inference_mode():
            for text in texts:
                logits = []
                for hypothesis in HYPOTHESES:
                    batch = tokenizer(
                        text,
                        hypothesis,
                        padding=True,
                        truncation='only_first',
                        max_length=MAX_LENGTH,
                        return_tensors='pt',
                    )
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    logits.append(model(**batch).logits[:, entailment[0]].float().cpu().item())
                all_scores.append(torch.softmax(torch.tensor(logits), dim=0).numpy())
        scores = np.asarray(all_scores, dtype=np.float64)
        method = 'zero_shot_nli'
    else:
        tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=False)
        load_options = {}
        if config.model_type == 'modernbert':
            config.reference_compile = False
            load_options['attn_implementation'] = 'eager'
        model = AutoModel.from_pretrained(
            repo, config=config, trust_remote_code=False, **load_options
        ).to(DEVICE).eval()

        descriptions = pd.DataFrame({'uid': [f'__description_{i}' for i in range(len(HYPOTHESES))], 'text': HYPOTHESES})
        all_texts = texts + descriptions['text'].tolist()
        batched = tokenizer(
            all_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_special_tokens_mask=True,
            return_tensors='pt',
        )
        batched = {k: v.to(DEVICE) for k, v in batched.items()}
        special = batched.pop('special_tokens_mask').to(DEVICE)
        with torch.inference_mode():
            hidden = model(**batched).last_hidden_state
        mask = (batched['attention_mask'] * (1 - special)).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
        embeddings = pooled.cpu().numpy()

        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        if np.any(norms < 1e-12):
            raise ValueError('Embedding de norma zero; não é possível calcular semelhança.')
        unit = embeddings / norms
        scores = unit[:len(texts)] @ unit[len(texts):].T
        method = 'cosine_descriptions'

    elapsed = time.perf_counter() - start
    return scores, method, elapsed


def record_result(model_name, repo, variant, method, probabilities, seconds_per_email, train_seconds=None, checkpoint=None):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != len(LABELS):
        raise ValueError(f'Scores inválidos para {model_name}: forma esperada {(len(val_examples), len(LABELS))}, recebida {probabilities.shape}.')

    y_true = np.asarray([item['label'] for item in val_examples], dtype=object)
    predicted_index = probabilities.argmax(axis=1)
    predicted_labels = np.asarray(LABELS, dtype=object)[predicted_index]
    present_labels = [label for label in LABELS if label in set(y_true)]

    def safe_recall(label):
        if label not in set(y_true):
            return np.nan
        return recall_score(
            y_true,
            predicted_labels,
            labels=[label],
            average='macro',
            zero_division=0,
        )

    requests_as_spam = int(np.sum((y_true != 'SPAM') & (predicted_labels == 'SPAM')))
    spam_as_request = int(np.sum((y_true == 'SPAM') & (predicted_labels != 'SPAM')))

    metrics.append({
        'model': model_name,
        'repo': repo,
        'variant': variant,
        'method': method,
        'accuracy': accuracy_score(y_true, predicted_labels),
        'macro_f1_present_classes': f1_score(
            y_true,
            predicted_labels,
            labels=present_labels,
            average='macro',
            zero_division=0,
        ) if present_labels else np.nan,
        'recall_pedido_de_informacao': safe_recall('Pedido de Informação'),
        'recall_pedido_de_encomenda': safe_recall('Pedido de Encomenda'),
        'requests_as_spam': requests_as_spam,
        'spam_as_request': spam_as_request,
        'seconds_per_email': float(seconds_per_email),
        'train_seconds': train_seconds,
        'checkpoint': checkpoint,
    })

    for index, item in enumerate(val_examples):
        row = {
            'uid': item['uid'],
            'model': model_name,
            'variant': variant,
            'method': method,
            'predicted': predicted_labels[index],
            'score': float(probabilities[index, predicted_index[index]]),
        }
        for label_index, label in enumerate(LABELS):
            row[f'score_{label}'] = float(probabilities[index, label_index])
        predictions.append(row)

In [22]:
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer):
        self.rows = []
        for item in examples:
            row = tokenizer(item['text'], truncation=True, max_length=MAX_LENGTH)
            row['labels'] = LABELS.index(item['label'])
            self.rows.append(row)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]

metrics, predictions, failures = [], [], []
for name, repo in MODELS:
    print(f'\nA comparar {name} ({repo})...', flush=True)
    model = tokenizer = trainer = None
    model_dir = RUN_DIR / name.replace(' ', '_').replace('/', '_')
    try:
        baseline_scores, method, baseline_seconds = baseline_predict(repo, val_examples)
        record_result(name, repo, 'sem_fine_tuning', method, baseline_scores, baseline_seconds)
    except Exception as error:
        failures.append({'model': name, 'variant': 'sem_fine_tuning',
                         'error': f'{type(error).__name__}: {error}'})
        print(f'  Baseline falhou: {type(error).__name__}: {error}')
    try:
        config = AutoConfig.from_pretrained(repo, num_labels=len(LABELS),
            id2label=dict(enumerate(LABELS)), label2id={label: i for i, label in enumerate(LABELS)},
            trust_remote_code=False)
        load_options = {}
        if config.model_type == 'modernbert':
            config.reference_compile = False
            load_options['attn_implementation'] = 'eager'
        tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(repo, config=config,
            ignore_mismatched_sizes=True, trust_remote_code=False, **load_options)
        train_set = EmailDataset(train_examples, tokenizer)
        val_set = EmailDataset(val_examples, tokenizer)
        trainer = Trainer(
            model=model,
            args=TrainingArguments(output_dir=str(model_dir), num_train_epochs=EPOCHS,
                per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                gradient_accumulation_steps=2, learning_rate=LEARNING_RATE,
                save_strategy='no', report_to='none', seed=SEED, data_seed=SEED),
            train_dataset=train_set, data_collator=DataCollatorWithPadding(tokenizer),
            processing_class=tokenizer,
        )
        train_start = time.perf_counter()
        trainer.train()
        train_seconds = time.perf_counter() - train_start
        trainer.save_model(str(model_dir))
        tokenizer.save_pretrained(model_dir)
        predict_start = time.perf_counter()
        logits = trainer.predict(val_set).predictions
        predict_seconds = time.perf_counter() - predict_start
        probabilities = torch.softmax(torch.as_tensor(logits), dim=-1).numpy()
        record_result(name, repo, 'fine_tuned', 'sequence_classifier', probabilities,
                      predict_seconds, train_seconds, str(model_dir))
    except Exception as error:
        failures.append({'model': name, 'variant': 'fine_tuned',
                         'error': f'{type(error).__name__}: {error}'})
        print(f'  Fine-tuning falhou: {type(error).__name__}: {error}')
    finally:
        pd.DataFrame(metrics).to_csv(RUN_DIR / 'metrics.csv', index=False, encoding='utf-8-sig')
        pd.DataFrame(predictions).to_csv(RUN_DIR / 'predictions.csv', index=False, encoding='utf-8-sig')
        (RUN_DIR / 'failures.json').write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding='utf-8')
        del trainer, model, tokenizer
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        if torch.backends.mps.is_available(): torch.mps.empty_cache()



A comparar NorBERTo-large (Itau-Unibanco/NorBERTo-large)...


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at Itau-Unibanco/NorBERTo-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss



A comparar XLM-RoBERTa-base (FacebookAI/xlm-roberta-base)...


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at PORTULAN/albertina-900m-portuguese-ptpt-encoder and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss



A comparar BERTimbau Base (neuralmind/bert-base-portuguese-cased)...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


In [23]:
if not metrics:
    raise RuntimeError(f'Nenhuma abordagem concluiu. Consulta {RUN_DIR / "failures.json"}.')
comparison = pd.DataFrame(metrics).sort_values(['model', 'variant'])
summary_columns = [
    'model', 'variant', 'method', 'accuracy', 'macro_f1_present_classes',
    'recall_pedido_de_informacao', 'recall_pedido_de_encomenda',
    'requests_as_spam', 'spam_as_request', 'seconds_per_email',
    'train_seconds', 'checkpoint'
]
display(comparison[[col for col in summary_columns if col in comparison.columns]])
paired = comparison.pivot(index='model', columns='variant',
                          values=['accuracy', 'macro_f1_present_classes', 'seconds_per_email'])
if {'sem_fine_tuning', 'fine_tuned'}.issubset(comparison['variant']):
    for metric in ['accuracy', 'macro_f1_present_classes']:
        paired[(metric, 'ganho_fine_tuning')] = (
            paired[(metric, 'fine_tuned')] - paired[(metric, 'sem_fine_tuning')])
display(paired)
display(pd.DataFrame(predictions).pivot(index='uid', columns=['model', 'variant'], values='predicted'))
if failures:
    print(f'{len(failures)} execução(ões) falharam; detalhes em {RUN_DIR / "failures.json"}.')

print('Métricas e previsões:', RUN_DIR)
print('Pedido de Encomenda não está na validação; o seu recall é indefinido neste ensaio.')


,model,variant,method,accuracy,macro_f1_present_classes,recall_pedido_de_informacao,recall_pedido_de_encomenda,requests_as_spam,spam_as_request,seconds_per_email,train_seconds,checkpoint
7,Albertina 900M PT-PT,fine_tuned,sequence_classifier,0.833333,0.777778,1.0,NaN,0,1,0.438184,16.947954,/home/alex/Desktop/GlobalBrico/research/finetu...
6,Albertina 900M PT-PT,sem_fine_tuning,cosine_descriptions,0.666667,0.400000,1.0,NaN,0,2,1.945922,NaN,NaN
9,BERTimbau Base,fine_tuned,sequence_classifier,0.666667,0.400000,1.0,NaN,0,2,0.051379,2.144072,/home/alex/Desktop/GlobalBrico/research/finetu...
8,BERTimbau Base,sem_fine_tuning,cosine_descriptions,0.666667,0.400000,1.0,NaN,0,2,0.924516,NaN,NaN
1,NorBERTo-large,fine_tuned,sequence_classifier,0.666667,0.400000,1.0,NaN,0,2,0.198563,7.343409,/home/alex/Desktop/GlobalBrico/research/finetu...
0,NorBERTo-large,sem_fine_tuning,cosine_descriptions,0.333333,0.250000,0.0,NaN,4,0,9.980057,NaN,NaN
3,XLM-RoBERTa-base,fine_tuned,sequence_classifier,0.666667,0.400000,1.0,NaN,0,2,0.051505,3.639438,/home/alex/Desktop/GlobalBrico/research/finetu...
2,XLM-RoBERTa-base,sem_fine_tuning,cosine_descriptions,0.000000,0.000000,0.0,NaN,0,2,8.584628,NaN,NaN
5,XLM-RoBERTa-large-XNLI,fine_tuned,sequence_classifier,0.666667,0.400000,1.0,NaN,0,2,0.127134,7.861034,/home/alex/Desktop/GlobalBrico/research/finetu...
4,XLM-RoBERTa-large-XNLI,sem_fine_tuning,zero_shot_nli,0.000000,0.000000,0.0,NaN,0,2,14.335069,NaN,NaN


accuracy                 macro_f1_present_classes  \
variant                fine_tuned sem_fine_tuning               fine_tuned   
model                                                                        
Albertina 900M PT-PT     0.833333        0.666667                 0.777778   
BERTimbau Base           0.666667        0.666667                 0.400000   
NorBERTo-large           0.666667        0.333333                 0.400000   
XLM-RoBERTa-base         0.666667        0.000000                 0.400000   
XLM-RoBERTa-large-XNLI   0.666667        0.000000                 0.400000   

                                       seconds_per_email                  \
variant                sem_fine_tuning        fine_tuned sem_fine_tuning   
model                                                                      
Albertina 900M PT-PT              0.40          0.438184        1.945922   
BERTimbau Base                    0.40          0.051379        0.924516   
NorBERTo-large                    0.25          0.198563        9.980057   
XLM-RoBERTa-base                  0.00          0.051505        8.584628   
XLM-RoBERTa-large-XNLI            0.00          0.127134       14.335069   

                                accuracy macro_f1_present_classes  
variant                ganho_fine_tuning        ganho_fine_tuning  
model                                                              
Albertina 900M PT-PT            0.166667                 0.377778  
BERTimbau Base                  0.000000                 0.000000  
NorBERTo-large                  0.333333                 0.150000  
XLM-RoBERTa-base                0.666667                 0.400000  
XLM-RoBERTa-large-XNLI          0.666667                 0.400000

model    NorBERTo-large                           XLM-RoBERTa-base  \
variant sem_fine_tuning            fine_tuned      sem_fine_tuning   
uid                                                                  
22545              SPAM  Pedido de Informação  Pedido de Encomenda   
25980              SPAM  Pedido de Informação  Pedido de Encomenda   
26007              SPAM  Pedido de Informação  Pedido de Encomenda   
26057              SPAM  Pedido de Informação  Pedido de Encomenda   
26505              SPAM  Pedido de Informação  Pedido de Encomenda   
26591              SPAM  Pedido de Informação  Pedido de Encomenda   

model                         XLM-RoBERTa-large-XNLI                        \
variant            fine_tuned        sem_fine_tuning            fine_tuned   
uid                                                                          
22545    Pedido de Informação    Pedido de Encomenda  Pedido de Informação   
25980    Pedido de Informação    Pedido de Encomenda  Pedido de Informação   
26007    Pedido de Informação    Pedido de Encomenda  Pedido de Informação   
26057    Pedido de Informação    Pedido de Encomenda  Pedido de Informação   
26505    Pedido de Informação    Pedido de Encomenda  Pedido de Informação   
26591    Pedido de Informação    Pedido de Encomenda  Pedido de Informação   

model    Albertina 900M PT-PT                              BERTimbau Base  \
variant       sem_fine_tuning            fine_tuned       sem_fine_tuning   
uid                                                                         
22545    Pedido de Informação                  SPAM  Pedido de Informação   
25980    Pedido de Informação  Pedido de Informação  Pedido de Informação   
26007    Pedido de Informação  Pedido de Informação  Pedido de Informação   
26057    Pedido de Informação  Pedido de Informação  Pedido de Informação   
26505    Pedido de Informação  Pedido de Informação  Pedido de Informação   
26591    Pedido de Informação  Pedido de Informação  Pedido de Informação   

model                          
variant            fine_tuned  
uid                            
22545    Pedido de Informação  
25980    Pedido de Informação  
26007    Pedido de Informação  
26057    Pedido de Informação  
26505    Pedido de Informação  
26591    Pedido de Informação

Métricas e previsões: /home/alex/Desktop/GlobalBrico/research/finetuning_comparison/20260914T150516Z
Pedido de Encomenda não está na validação; o seu recall é indefinido neste ensaio.
